In [74]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
!pip install xgboost lightgbm
import xgboost as xgb
import lightgbm as lgb

In [75]:
df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')

In [76]:
df.drop('id', axis=1, inplace=True)

In [77]:
columns_to_drop = ['gender', 'PhoneService', 'MultipleLines', 'StreamingTV', 'StreamingMovies']
df.drop(columns=columns_to_drop, inplace=True)
print(f"Dropped columns: {columns_to_drop}")

Dropped columns: ['gender', 'PhoneService', 'MultipleLines', 'StreamingTV', 'StreamingMovies']


In [78]:
object_cols_remaining = df.select_dtypes(include='object').columns.tolist()
categorical_cols_for_encoding = [col for col in object_cols_remaining if col != 'Churn']

if 'SeniorCitizen' not in categorical_cols_for_encoding and 'SeniorCitizen' in df.columns:
    categorical_cols_for_encoding.append('SeniorCitizen')

print("Categorical Columns for Encoding (excluding 'Churn'):")
print(categorical_cols_for_encoding)

Categorical Columns for Encoding (excluding 'Churn'):
['Partner', 'Dependents', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'SeniorCitizen']


In [79]:
# Apply Label Encoding to binary categorical columns
for col in categorical_cols_for_encoding:
    if df[col].nunique() == 2:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        print(f"Label Encoded: {col}")

# Apply One-Hot Encoding to categorical columns with more than two unique values
df = pd.get_dummies(df, columns=[col for col in categorical_cols_for_encoding if df[col].nunique() > 2], drop_first=True)
print("One-Hot Encoded remaining categorical columns.")

df.head()

Label Encoded: Partner
Label Encoded: Dependents
Label Encoded: PaperlessBilling
Label Encoded: SeniorCitizen
One-Hot Encoded remaining categorical columns.


,SeniorCitizen,Partner,Dependents,tenure,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,InternetService_Fiber optic,InternetService_No,...,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,1,29,1,60.10,1653.85,No,False,False,...,False,False,True,False,True,True,False,False,False,True
1,0,1,1,58,0,69.50,3778.20,No,False,False,...,True,False,False,False,True,False,True,True,False,False
2,0,1,0,58,1,100.40,5841.35,No,True,False,...,True,False,False,False,False,False,False,False,True,False
3,0,0,0,1,1,69.70,70.70,Yes,True,False,...,False,False,False,False,False,False,False,False,True,False
4,0,0,0,1,1,70.45,70.45,Yes,True,False,...,False,False,False,False,False,False,False,False,True,False


In [80]:
scaler = StandardScaler()
df['tenure_scaled'] = scaler.fit_transform(df[['tenure']])
df['MonthlyCharges_scaled'] = scaler.fit_transform(df[['MonthlyCharges']])
df['TotalCharges_scaled'] = scaler.fit_transform(df[['TotalCharges']])

columns_to_drop_original_numerical = ['tenure', 'MonthlyCharges','TotalCharges']
df.drop(columns=columns_to_drop_original_numerical, inplace=True)
print(f"Dropped original columns: {columns_to_drop_original_numerical}")
df.head()

Dropped original columns: ['tenure', 'MonthlyCharges', 'TotalCharges']


,SeniorCitizen,Partner,Dependents,PaperlessBilling,Churn,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No internet service,...,TechSupport_No internet service,TechSupport_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_scaled,MonthlyCharges_scaled,TotalCharges_scaled
0,0,1,1,1,No,False,False,False,True,False,...,False,True,True,False,False,False,True,-0.302342,-0.185604,-0.357076
1,0,1,1,0,No,False,False,False,True,False,...,False,True,False,True,True,False,False,0.854793,0.116964,0.545399
2,0,1,0,1,No,True,False,False,False,False,...,False,False,False,False,False,True,False,0.854793,1.111575,1.421875
3,0,0,0,1,Yes,True,False,False,False,False,...,False,False,False,False,False,True,False,-1.419575,0.123402,-1.029637
4,0,0,0,1,Yes,True,False,False,False,False,...,False,False,False,False,False,True,False,-1.419575,0.147543,-1.029743


In [81]:
le = LabelEncoder()
df['Churn'] = le.fit_transform(df['Churn'])
print("Encoded 'Churn' column:")
print(df['Churn'].value_counts())
df.head()

Encoded 'Churn' column:
Churn
0    460377
1    133817
Name: count, dtype: int64


,SeniorCitizen,Partner,Dependents,PaperlessBilling,Churn,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No internet service,...,TechSupport_No internet service,TechSupport_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_scaled,MonthlyCharges_scaled,TotalCharges_scaled
0,0,1,1,1,0,False,False,False,True,False,...,False,True,True,False,False,False,True,-0.302342,-0.185604,-0.357076
1,0,1,1,0,0,False,False,False,True,False,...,False,True,False,True,True,False,False,0.854793,0.116964,0.545399
2,0,1,0,1,0,True,False,False,False,False,...,False,False,False,False,False,True,False,0.854793,1.111575,1.421875
3,0,0,0,1,1,True,False,False,False,False,...,False,False,False,False,False,True,False,-1.419575,0.123402,-1.029637
4,0,0,0,1,1,True,False,False,False,False,...,False,False,False,False,False,True,False,-1.419575,0.147543,-1.029743


In [82]:
X = df.drop('Churn', axis=1)
y = df['Churn']
print("Features (X) and target (y) have been separated.")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

Features (X) and target (y) have been separated.
X shape: (594194, 22)
y shape: (594194,)


In [83]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Dataset split into training and testing sets.")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

Dataset split into training and testing sets.
X_train shape: (475355, 22)
X_test shape: (118839, 22)
y_train shape: (475355,)
y_test shape: (118839,)


# **Model Training**

In [84]:
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    eval_metric='logloss',
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42,
    device='cuda'
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=20
)

[0]	validation_0-logloss:0.52121
[20]	validation_0-logloss:0.39117
[40]	validation_0-logloss:0.34338
[60]	validation_0-logloss:0.32241
[80]	validation_0-logloss:0.31268
[100]	validation_0-logloss:0.30791
[120]	validation_0-logloss:0.30535
[140]	validation_0-logloss:0.30391
[160]	validation_0-logloss:0.30305
[180]	validation_0-logloss:0.30236
[200]	validation_0-logloss:0.30183
[220]	validation_0-logloss:0.30138
[240]	validation_0-logloss:0.30103
[260]	validation_0-logloss:0.30068
[280]	validation_0-logloss:0.30040
[300]	validation_0-logloss:0.30019
[320]	validation_0-logloss:0.29993
[340]	validation_0-logloss:0.29978
[360]	validation_0-logloss:0.29953
[380]	validation_0-logloss:0.29936
[400]	validation_0-logloss:0.29923
[420]	validation_0-logloss:0.29907
[440]	validation_0-logloss:0.29897
[460]	validation_0-logloss:0.29887
[480]	validation_0-logloss:0.29879
[499]	validation_0-logloss:0.29871


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, ...)

In [85]:
y_pred_xgb = xgb_model.predict(X_test)

accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
print(f"XGBoost Classifier Accuracy: {accuracy_xgb:.4f}")

print("\nClassification Report (XGBoost):\n")
print(classification_report(y_test, y_pred_xgb))

print("\nConfusion Matrix (XGBoost):\n")
print(confusion_matrix(y_test, y_pred_xgb))

XGBoost Classifier Accuracy: 0.8607

Classification Report (XGBoost):

              precision    recall  f1-score   support

           0       0.90      0.92      0.91     92076
           1       0.71      0.64      0.68     26763

    accuracy                           0.86    118839
   macro avg       0.81      0.78      0.79    118839
weighted avg       0.86      0.86      0.86    118839


Confusion Matrix (XGBoost):

[[85098  6978]
 [ 9572 17191]]


In [86]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    device="gpu"
)

lgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[
        lgb.log_evaluation(50)
    ]
)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 621
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 22
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 12 dense feature groups (5.44 MB) transferred to GPU in 0.008018 secs. 1 sparse feature groups
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225209 -> initscore=-1.235567
[LightGBM] [Info] Start training from score -1.235567
[50]	valid_0's binary_logloss: 0.332638
[100]	valid_0's binary_logloss: 0.308601
[150]	valid_0's binary_logloss: 0.303947

LGBMClassifier(colsample_bytree=0.8, device='gpu', learning_rate=0.03,
               max_depth=6, n_estimators=500, random_state=42, subsample=0.8)

In [87]:
y_pred_lgb = lgb_model.predict(X_test)

accuracy_lgb = accuracy_score(y_test, y_pred_lgb)
print(f"LightGBM Classifier Accuracy: {accuracy_lgb:.4f}")

print("\nClassification Report (LightGBM):\n")
print(classification_report(y_test, y_pred_lgb))

print("\nConfusion Matrix (LightGBM):\n")
print(confusion_matrix(y_test, y_pred_lgb))

LightGBM Classifier Accuracy: 0.8603

Classification Report (LightGBM):

              precision    recall  f1-score   support

           0       0.90      0.92      0.91     92076
           1       0.71      0.64      0.67     26763

    accuracy                           0.86    118839
   macro avg       0.80      0.78      0.79    118839
weighted avg       0.86      0.86      0.86    118839


Confusion Matrix (LightGBM):

[[85056  7020]
 [ 9576 17187]]


In [88]:
!pip install optuna

In [89]:
import optuna
from sklearn.metrics import roc_auc_score

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

def objective_xgb(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 500, 700),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 3),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 3),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 3),
        "tree_method": "hist",
        "device": "cuda",
        "eval_metric": "auc"
    }

    model = xgb.XGBClassifier(**params)

    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    preds = model.predict_proba(X_val)[:,1]

    auc = roc_auc_score(y_val, preds)

    return auc

In [90]:
study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=50)

print(study_xgb.best_params)

[I 2026-03-12 11:36:29,878] A new study created in memory with name: no-name-d54c75bf-c237-49f9-8f8d-23686f1af447
[I 2026-03-12 11:36:32,185] Trial 0 finished with value: 0.9153976365077733 and parameters: {'n_estimators': 557, 'max_depth': 7, 'learning_rate': 0.07765051297154407, 'subsample': 0.894065251143416, 'colsample_bytree': 0.8913949986215594, 'gamma': 2.903723571257475, 'reg_alpha': 0.7948067095712047, 'reg_lambda': 2.5521722941108957}. Best is trial 0 with value: 0.9153976365077733.
[I 2026-03-12 11:36:34,757] Trial 1 finished with value: 0.9149205407768859 and parameters: {'n_estimators': 601, 'max_depth': 4, 'learning_rate': 0.040352556348395296, 'subsample': 0.7159820813640239, 'colsample_bytree': 0.6231305667298837, 'gamma': 2.9156167353052664, 'reg_alpha': 2.753047434979088, 'reg_lambda': 0.8843635853233348}. Best is trial 0 with value: 0.9153976365077733.
[I 2026-03-12 11:36:37,642] Trial 2 finished with value: 0.9154780254960981 and parameters: {'n_estimators': 586, 'm

{'n_estimators': 612, 'max_depth': 4, 'learning_rate': 0.0969477093532691, 'subsample': 0.8144380295999682, 'colsample_bytree': 0.7100090345464319, 'gamma': 0.064797908848089, 'reg_alpha': 0.14146079027745373, 'reg_lambda': 2.994978601878179}


In [91]:
def objective_lgb(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 500, 700),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "num_leaves": trial.suggest_int("num_leaves", 20, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 3),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 3),
        "random_state": 42
    }

    model = lgb.LGBMClassifier(**params)

    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(10)]
    )

    preds = model.predict_proba(X_val)[:,1]

    auc = roc_auc_score(y_val, preds)

    return auc

In [92]:
study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(objective_lgb, n_trials=50)

print(study_lgb.best_params)

[I 2026-03-12 11:38:52,282] A new study created in memory with name: no-name-4fab03a1-8617-4532-ae33-835e3c445a1b


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 85770, number of negative: 294514
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016091 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 621
[LightGBM] [Info] Number of data points in the train set: 380284, number of used features: 22
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225542 -> initscore=-1.233657
[LightGBM] [Info] Start training from score -1.233657
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 10 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

[I 2026-03-12 11:39:06,859] Trial 0 finished with value: 0.9159797146241123 and parameters: {'n_estimators': 606, 'learning_rate': 0.08133335248817634, 'max_depth': 4, 'num_leaves': 29, 'subsample': 0.6003035065558212, 'colsample_bytree': 0.7184367013374909, 'reg_alpha': 0.8743908337797378, 'reg_lambda': 0.9831039346607479}. Best is trial 0 with value: 0.9159797146241123.


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 85770, number of negative: 294514
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016125 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 621
[LightGBM] [Info] Number of data points in the train set: 380284, number of used features: 22
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225542 -> initscore=-1.233657
[LightGBM] [Info] Start training from score -1.233657
Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[448]	valid_0's auc: 0.916065	valid_0's binary_logloss: 0.297315


[I 2026-03-12 11:39:18,831] Trial 1 finished with value: 0.9160650094295734 and parameters: {'n_estimators': 659, 'learning_rate': 0.10600646872527207, 'max_depth': 6, 'num_leaves': 20, 'subsample': 0.8140111177132926, 'colsample_bytree': 0.6672668442104374, 'reg_alpha': 1.2044936771211572, 'reg_lambda': 0.25883866673836464}. Best is trial 1 with value: 0.9160650094295734.


...

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 85770, number of negative: 294514
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016758 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 621
[LightGBM] [Info] Number of data points in the train set: 380284, number of used features: 22
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225542 -> initscore=-1.233657
[LightGBM] [Info] Start training from score -1.233657
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 10 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

[I 2026-03-12 11:46:48,429] Trial 48 finished with value: 0.9158012611879407 and parameters: {'n_estimators': 589, 'learning_rate': 0.11901164558800628, 'max_depth': 5, 'num_leaves': 73, 'subsample': 0.9960209986777099, 'colsample_bytree': 0.847367963558693, 'reg_alpha': 0.7330173620713493, 'reg_lambda': 1.917280109031513}. Best is trial 36 with value: 0.9161373878224691.


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 85770, number of negative: 294514
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015734 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 621
[LightGBM] [Info] Number of data points in the train set: 380284, number of used features: 22
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225542 -> initscore=-1.233657
[LightGBM] [Info] Start training from score -1.233657
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 10 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

[I 2026-03-12 11:46:58,711] Trial 49 finished with value: 0.9159086577995812 and parameters: {'n_estimators': 699, 'learning_rate': 0.10521423033004626, 'max_depth': 5, 'num_leaves': 31, 'subsample': 0.9554838574951378, 'colsample_bytree': 0.7620828866009204, 'reg_alpha': 1.2695339689357636, 'reg_lambda': 1.6689036190699396}. Best is trial 36 with value: 0.9161373878224691.


{'n_estimators': 614, 'learning_rate': 0.11643873097312782, 'max_depth': 4, 'num_leaves': 42, 'subsample': 0.9231031441209556, 'colsample_bytree': 0.7903598106967457, 'reg_alpha': 0.6184532053195697, 'reg_lambda': 0.818225839448754}


In [93]:
best_xgb = xgb.XGBClassifier(
    **study_xgb.best_params,
    tree_method="hist",
    device="cuda",
    eval_metric="auc",
    random_state=42
)

best_xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7100090345464319, device='cuda',
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='auc', feature_types=None, feature_weights=None,
              gamma=0.064797908848089, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.0969477093532691,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=612, n_jobs=None,
              num_parallel_tree=None, ...)

In [94]:
best_lgb = lgb.LGBMClassifier(
    **study_lgb.best_params,
    random_state=42
)

best_lgb.fit(X_train, y_train)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.020341 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 621
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 22
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225209 -> initscore=-1.235567
[LightGBM] [Info] Start training from score -1.235567
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

LGBMClassifier(colsample_bytree=0.7903598106967457,
               learning_rate=0.11643873097312782, max_depth=4, n_estimators=614,
               num_leaves=42, random_state=42, reg_alpha=0.6184532053195697,
               reg_lambda=0.818225839448754, subsample=0.9231031441209556)

In [95]:
from sklearn.metrics import accuracy_score

# XGBoost
xgb_preds = best_xgb.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_preds)

print("XGBoost Accuracy:", xgb_acc)


# LightGBM
lgb_preds = best_lgb.predict(X_test)
lgb_acc = accuracy_score(y_test, lgb_preds)

print("LightGBM Accuracy:", lgb_acc)

XGBoost Accuracy: 0.8609547370812612
LightGBM Accuracy: 0.8617120642213415


In [96]:
prob_xgb = best_xgb.predict_proba(X_test)[:,1]
prob_lgb = best_lgb.predict_proba(X_test)[:,1]

In [97]:
ensemble_prob = 0.3 * prob_xgb + 0.7 * prob_lgb

ensemble_preds = (ensemble_prob > 0.5).astype(int)

ensemble_acc = accuracy_score(y_test, ensemble_preds)

print("Ensemble Accuracy:", ensemble_acc)

Ensemble Accuracy: 0.8617204789673424
